In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [82]:
import yfinance as yf

# READ Listings from BSE and NSE

In [83]:
import pandas as pd

# Load the file
# index_col=False prevents Pandas from using the first columns as an index
# due to the mismatch in column counts caused by trailing commas.
df = pd.read_csv('/content/BSE_LIST.csv', index_col=False)

# Identify and drop the extra columns created by trailing commas
# These columns usually have no name (Unnamed: X) or contain only NaN values
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# The snippet showed some columns might be completely empty due to the shift.
# We ensure we only keep the 9 columns defined in the header.
header_columns = [
    'Security Code', 'Issuer Name', 'Security Id',
    'Security Name', 'Status', 'Group', 'Face Value',
    'ISIN No', 'Instrument'
]

# Re-index to ensure we have exactly these columns and they are in order
df = df[header_columns]

# Clean trailing spaces from columns like 'Group' which had 'A ' instead of 'A'
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

print("Cleaned BSE Data Preview:")
df.head()

Cleaned BSE Data Preview:


/tmp/ipykernel_423/3857880396.py:6: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv('/content/BSE_LIST.csv', index_col=False)


,Security Code,Issuer Name,Security Id,Security Name,Status,Group,Face Value,ISIN No,Instrument
0,500002,ABB India Limited,ABB,ABB India Ltd,Active,A,2.0,INE117A01022,Equity
1,500003,Aegis Logistics Ltd.,AEGISLOG,Aegis Logistics Ltd,Active,A,1.0,INE208C01025,Equity
2,500008,Amara Raja Energy & Mobility Limited,ARE&M,Amara Raja Energy & Mobility Ltd,Active,A,1.0,INE885A01032,Equity
3,500009,Ambalal Sarabhai Enterprise Ltd.,AMBALALSA,Ambalal Sarabhai Enterprises Ltd,Active,X,10.0,INE432A01017,Equity
4,500012,Andhra Petrochemicals Ltd.,ANDHRAPET,Andhra Petrochemicals Ltd,Active,X,10.0,INE714B01016,Equity


In [84]:
df_BSE = df.copy()
print(f"Dataframe has {df_BSE.shape[0]} Rows and {df_BSE.shape[1]} Columns")
df_BSE.head(5)

Dataframe has 4843 Rows and 9 Columns


,Security Code,Issuer Name,Security Id,Security Name,Status,Group,Face Value,ISIN No,Instrument
0,500002,ABB India Limited,ABB,ABB India Ltd,Active,A,2.0,INE117A01022,Equity
1,500003,Aegis Logistics Ltd.,AEGISLOG,Aegis Logistics Ltd,Active,A,1.0,INE208C01025,Equity
2,500008,Amara Raja Energy & Mobility Limited,ARE&M,Amara Raja Energy & Mobility Ltd,Active,A,1.0,INE885A01032,Equity
3,500009,Ambalal Sarabhai Enterprise Ltd.,AMBALALSA,Ambalal Sarabhai Enterprises Ltd,Active,X,10.0,INE432A01017,Equity
4,500012,Andhra Petrochemicals Ltd.,ANDHRAPET,Andhra Petrochemicals Ltd,Active,X,10.0,INE714B01016,Equity


In [85]:
df_BSE.columns

Index(['Security Code', 'Issuer Name', 'Security Id', 'Security Name',
       'Status', 'Group', 'Face Value', 'ISIN No', 'Instrument'],
      dtype='object')

In [86]:
df_BSE = df_BSE.rename(columns={
    "ISIN No": "ISIN",
    "Security Name": "Company_Name",
    "Face Value": "Face_Value",
    "Security Id": "Symbol"
})
df_BSE.columns

Index(['Security Code', 'Issuer Name', 'Symbol', 'Company_Name', 'Status',
       'Group', 'Face_Value', 'ISIN', 'Instrument'],
      dtype='object')

In [87]:
df_BSE.head(5)

,Security Code,Issuer Name,Symbol,Company_Name,Status,Group,Face_Value,ISIN,Instrument
0,500002,ABB India Limited,ABB,ABB India Ltd,Active,A,2.0,INE117A01022,Equity
1,500003,Aegis Logistics Ltd.,AEGISLOG,Aegis Logistics Ltd,Active,A,1.0,INE208C01025,Equity
2,500008,Amara Raja Energy & Mobility Limited,ARE&M,Amara Raja Energy & Mobility Ltd,Active,A,1.0,INE885A01032,Equity
3,500009,Ambalal Sarabhai Enterprise Ltd.,AMBALALSA,Ambalal Sarabhai Enterprises Ltd,Active,X,10.0,INE432A01017,Equity
4,500012,Andhra Petrochemicals Ltd.,ANDHRAPET,Andhra Petrochemicals Ltd,Active,X,10.0,INE714B01016,Equity


In [88]:
df_BSE.to_csv('BSE_LIST_CLEANED.csv', index=False)

In [89]:
# Load NSE List
nse_df = pd.read_csv('/content/NSE_LIST.csv')

# 1. Clean Column Names
# This removes the leading spaces like ' SERIES' -> 'SERIES'
nse_df.columns = nse_df.columns.str.strip()

# 2. Standardize Naming Convention
# We map these to match the BSE-style or a neutral unified style
nse_map = {
    'SYMBOL': 'Symbol',
    'NAME OF COMPANY': 'Company_Name',
    'ISIN NUMBER': 'ISIN',
    'FACE VALUE': 'Face_Value'
}
nse_df = nse_df.rename(columns=nse_map)

# 3. Handle Data Types
# Ensure Symbol and ISIN are strings and stripped of any internal padding
for col in ['Symbol', 'ISIN']:
    nse_df[col] = nse_df[col].astype(str).str.strip()

# 4. Filter for merge-ready columns
# We only keep what is necessary for a unified view
nse_cleaned = nse_df[['Symbol', 'Company_Name', 'ISIN', 'Face_Value']]

print("Cleaned NSE Data Preview:")
nse_cleaned.head()

Cleaned NSE Data Preview:


,Symbol,Company_Name,ISIN,Face_Value
0,20MICRONS,20 Microns Limited,INE144J01027,5
1,21STCENMGM,21st Century Management Services Limited,INE253B01015,10
2,360ONE,360 ONE WAM LIMITED,INE466L01038,1
3,3IINFOLTD,3i Infotech Limited,INE748C01038,10
4,3MINDIA,3M India Limited,INE470A01017,10


In [90]:
nse_cleaned.columns

Index(['Symbol', 'Company_Name', 'ISIN', 'Face_Value'], dtype='object')

In [91]:
df_BSE.isna().sum()

,0
Security Code,0
Issuer Name,0
Symbol,0
Company_Name,0
Status,0
Group,0
Face_Value,0
ISIN,0
Instrument,0


In [92]:
nse_cleaned.isna().sum()

,0
Symbol,0
Company_Name,0
ISIN,0
Face_Value,0


In [93]:
# 1. Prepare BSE Data
# Drop the single row where ISIN is missing to prevent a bad merge
# Replace text-based nulls/dashes with actual np.nan in both dataframes
df_BSE['ISIN'] = df_BSE['ISIN'].replace(['-', 'nan', 'NA', 'N/A', ''], np.nan)
nse_cleaned['ISIN'] = nse_cleaned['ISIN'].replace(['-', 'nan', 'NA', 'N/A', ''], np.nan)

# 1. Prepare BSE Data (Now dropna will catch the dash!)
bse_merge_df = df_BSE.dropna(subset=['ISIN'])[['ISIN', 'Symbol', 'Company_Name', 'Security Code', 'Face_Value']].copy()
bse_merge_df.rename(columns={'Symbol': 'BSE_Symbol', 'Company_Name': 'Company_Name_BSE', 'Face_Value': 'Face_Value_BSE'}, inplace=True)
bse_merge_df['ON_BSE'] = 1

# 2. Prepare NSE Data
nse_merge_df = nse_cleaned.dropna(subset=['ISIN'])[['ISIN', 'Symbol', 'Company_Name', 'Face_Value']].copy()
nse_merge_df.rename(columns={'Symbol': 'NSE_Symbol', 'Company_Name': 'Company_Name_NSE', 'Face_Value': 'Face_Value_NSE'}, inplace=True)
nse_merge_df['ON_NSE'] = 1

# ... Proceed with Step 3 (Outer Merge) as before ...

# 3. Outer Merge on ISIN
merged_df = pd.merge(bse_merge_df, nse_merge_df, on='ISIN', how='outer')

# 4. Clean up the merged flags (fill NaN with 0 and convert to integer)
merged_df['ON_BSE'] = merged_df['ON_BSE'].fillna(0).astype(int)
merged_df['ON_NSE'] = merged_df['ON_NSE'].fillna(0).astype(int)

# 5. Consolidate overlapping columns (Company Name and Face Value)
# If a company is only on BSE, the NSE fields will be NaN. We fill those gaps seamlessly.
merged_df['Company_Name'] = merged_df['Company_Name_NSE'].fillna(merged_df['Company_Name_BSE'])
merged_df['Face_Value'] = merged_df['Face_Value_NSE'].fillna(merged_df['Face_Value_BSE'])

# Drop the temporary split columns to keep the dataframe clean
merged_df.drop(columns=['Company_Name_BSE', 'Company_Name_NSE', 'Face_Value_BSE', 'Face_Value_NSE'], inplace=True)

# Format the remaining empty spots nicely
merged_df['BSE_Symbol'] = merged_df['BSE_Symbol'].fillna('N/A')
merged_df['NSE_Symbol'] = merged_df['NSE_Symbol'].fillna('N/A')
merged_df['Security Code'] = merged_df['Security Code'].fillna('N/A')

# 6. Reorder columns for a clean final dataset
final_columns = [
    'ISIN', 'Company_Name', 'BSE_Symbol', 'NSE_Symbol',
    'Security Code', 'Face_Value', 'ON_BSE', 'ON_NSE'
]
merged_df = merged_df[final_columns]

# Print summary stats
print(f"Total Unique Companies (ISINs): {len(merged_df)}")
print(f"Listed on Both (ON_BSE=1 & ON_NSE=1): {len(merged_df[(merged_df['ON_BSE']==1) & (merged_df['ON_NSE']==1)])}")
print(f"BSE Only (ON_BSE=1 & ON_NSE=0): {len(merged_df[(merged_df['ON_BSE']==1) & (merged_df['ON_NSE']==0)])}")
print(f"NSE Only (ON_BSE=0 & ON_NSE=1): {len(merged_df[(merged_df['ON_BSE']==0) & (merged_df['ON_NSE']==1)])}")

# Preview the final result
merged_df.head(10)

Total Unique Companies (ISINs): 4974
Listed on Both (ON_BSE=1 & ON_NSE=1): 2115
BSE Only (ON_BSE=1 & ON_NSE=0): 2726
NSE Only (ON_BSE=0 & ON_NSE=1): 133


/tmp/ipykernel_423/3490541904.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nse_cleaned['ISIN'] = nse_cleaned['ISIN'].replace(['-', 'nan', 'NA', 'N/A', ''], np.nan)


,ISIN,Company_Name,BSE_Symbol,NSE_Symbol,Security Code,Face_Value,ON_BSE,ON_NSE
0,IN90GGO01013,KRISHIVAL FOODS Ltd,KRISHPP,N/A,890232.0,10.0,1,0
1,IN90NN701011,Allcargo Terminals Ltd,ATLPP,N/A,890228.0,2.0,1,0
2,IN9161F01019,ANIRIT VENTURES Ltd,ANIRITPP,N/A,890231.0,10.0,1,0
3,IN9170U01035,Suraj Industries Ltd,SURAJINDPP,N/A,890226.0,10.0,1,0
4,IN9175A01010,Jain Irrigation Systems Limited,JISLDVREQS,JISLDVREQS,570004.0,2.0,1,1
5,IN9237C01014,KATI PATANG LIFESTYLE Ltd,PATANGPP,N/A,890220.0,10.0,1,0
6,IN9273A01047,Aplab Ltd,APLABPP,N/A,890217.0,10.0,1,0
7,IN9428B01029,PVV Infra Ltd,PVVIPP,N/A,890233.0,5.0,1,0
8,IN9564C01011,Yarn Syndicate Ltd,YARNPP,N/A,890197.0,10.0,1,0
9,IN9572J01019,Spandana Sphoorty Financial Ltd,SSFLPP,N/A,890221.0,10.0,1,0


In [94]:
merged_df.shape

(4974, 8)

In [80]:
merged_df.to_csv("INDIA_LIST.csv", index=False)